# 📘 Notebook 2b: Many Models per LOT_ID (MMT)

## Overview

**Alternative to Notebook 2.** Rather than training one global DNN, train **one DNN per `LOT_ID`** in parallel using Snowflake's `ManyModelTraining` (MMT) API. Use this pattern when per-entity models materially outperform a single global model.

### When to Consider MMT

- Per-entity models outperform a global model (each lot, customer, or SKU has distinct behavior)
- You have 100s to 10,000s of partitions — too many to train by hand
- Training takes more than a few seconds per partition, so framework overhead is amortized
- You want to persist each per-partition model and re-score on a schedule

### What's Covered

| Section | Topic |
|---|---|
| 1 | Setup — imports, session, constants |
| 2 | Load dataset + join `LOT_ID` partition key |
| 3 | Shared training recipe (same architecture as Notebook 2) |
| 4 | Profile one lot to project wall-clock |
| 5 | Scale the Container Runtime cluster |
| 6 | Export partitioned parquet to stage |
| 7 | Run MMT via `run_from_stage` |
| 8 | Inspect per-lot results + test metrics |
| 9 | Run distributed inference (MMI) |
| 10 | Summary |

### Handoff from Notebook 1

Loads `WAFER_YIELD_TRAINING_DATASET v1_train` then joins in `LOT_ID` from `WAFER_PROCESS_DATA`. Splits each lot 80/20 internally so every per-lot model reports held-out metrics — comparable to Notebook 2.

### Handoff to Downstream

Writes predictions to `WAFER_YIELD_DEMO.INFERENCE.LOT_PREDICTIONS`. Per-lot models are persisted at the model stage under `run_id` and can be re-scored without retraining via `ManyModelInference`.


## 📘 Section 1 — Setup

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import time
import numpy as np
import pandas as pd
from datetime import datetime

import torch
import torch.nn as nn

from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, count as sf_count, avg
from snowflake.snowpark.context import get_active_session

from snowflake.ml import dataset
from snowflake.ml.data import DataConnector
from snowflake.ml.modeling.distributors.many_model import (
    ManyModelTraining, ManyModelInference, TorchSerde
)
from snowflake.ml.modeling.distributors.distributed_partition_function.entities import ExecutionOptions
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes

from sklearn.preprocessing import StandardScaler

print(f"PyTorch: {torch.__version__}   |   CUDA available: {torch.cuda.is_available()}")


In [ ]:
# ============================================================================
# SESSION + CONSTANTS
# ============================================================================
session = get_active_session()
session.sql("USE WAREHOUSE WAFER_DEMO_WH").collect()

# Data constants
DATASET_NAME     = "WAFER_YIELD_TRAINING_DATASET"
DATASET_VERSION  = "v1"

# Training infrastructure
MODEL_STAGE      = "WAFER_YIELD_DEMO.RAW_DATA.MANY_MODELS_STAGE"
DATA_STAGE       = "WAFER_YIELD_DEMO.RAW_DATA.LOT_DATA_STAGE"
TRAINING_STAGING_TABLE = "WAFER_YIELD_DEMO.RAW_DATA.LOT_TRAINING_STAGING"
PREDICTIONS_TABLE = "WAFER_YIELD_DEMO.INFERENCE.LOT_PREDICTIONS"

# Partition + run config
PARTITION_COL    = "LOT_ID"
RUN_ID           = f"wafer_yield_lot_models_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Session OK | WH: {session.get_current_warehouse()}")
print(f"Run ID: {RUN_ID}")


## 📘 Section 2 — Load Data + Join LOT_ID

The ML Dataset from Notebook 01 doesn't carry `LOT_ID` — we join it in from raw process data and materialize a staging table so we can export partitioned parquet for MMT.


In [ ]:
# Load ML Dataset from Notebook 01
wafer_dataset = dataset.load_dataset(session, DATASET_NAME, DATASET_VERSION)
features_labels_sf = wafer_dataset.read.to_snowpark_dataframe()

# Join LOT_ID from raw process data
lot_lookup = session.sql('''
    SELECT DISTINCT WAFER_ID, LOT_ID
    FROM WAFER_YIELD_DEMO.RAW_DATA.WAFER_PROCESS_DATA
''')
training_data_sf = features_labels_sf.join(lot_lookup, "WAFER_ID")

# Materialize into a staging table (so COPY INTO @stage PARTITION BY LOT_ID can run cleanly)
training_data_sf.write.mode("overwrite").save_as_table(TRAINING_STAGING_TABLE)
training_staging = session.table(TRAINING_STAGING_TABLE)

total_rows = training_staging.count()
total_lots = training_staging.select(PARTITION_COL).distinct().count()
avg_per_lot = total_rows // total_lots
print(f"Training staging: {total_rows:,} rows across {total_lots:,} lots  |  avg ~{avg_per_lot:,} rows/lot")
training_staging.limit(5).show()


In [ ]:
# Identify feature and label columns (used by training + inference recipes)
all_columns = training_staging.columns
EXCLUDE_COLS = ['WAFER_ID', 'LOT_ID', 'YIELD_GOOD', 'YIELD_SCORE', 'DOMINANT_DEFECT_TYPE']
FEATURE_COLS = [c for c in all_columns if c.upper() not in [x.upper() for x in EXCLUDE_COLS]]
LABEL_COL = 'YIELD_GOOD'

print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS[:8]}{'...' if len(FEATURE_COLS) > 8 else ''}")
print(f"Label: {LABEL_COL}")


## 📘 Section 3 — Shared Training Recipe

One function — `train_lot_dnn_recipe(df)` — is the single source of truth for training logic. It's called by:

- The profile cell below (times one lot locally)
- The MMT training function (runs once per partition on a GPU worker)

Seeds are fixed (`torch.manual_seed(42)`) so results are reproducible.


In [ ]:
def train_lot_dnn_recipe(df_train, df_test=None, random_seed: int = 42):
    """Trains one DNN on one lot's training data. Returns (scaler, trained_model, metrics_dict).
    Architecture matches Notebook 02 so the global-vs-per-lot comparison is apples-to-apples.
    If df_test is provided, metrics are computed on TEST (non-leaky).
    """
    import torch, torch.nn as nn
    import numpy as np
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    torch.manual_seed(random_seed); np.random.seed(random_seed)

    exclude_cols = ['WAFER_ID', 'LOT_ID', 'FIRST_TIMESTAMP', 'YIELD_GOOD',
                    'YIELD_SCORE', 'DOMINANT_DEFECT_TYPE']
    feature_cols = [c for c in df_train.columns if c.upper() not in [x.upper() for x in exclude_cols]]

    X_train = df_train[feature_cols].fillna(0).values.astype('float32')
    y_train = df_train['YIELD_GOOD'].values.astype('float32')

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    # Architecture matches the global DNN in Notebook 02.
    input_dim = X_train.shape[1]
    model = nn.Sequential(
        nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(64, 32),         nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(32, 1),          nn.Sigmoid(),
    )

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    Xt = torch.tensor(X_train).to(device); yt = torch.tensor(y_train).unsqueeze(1).to(device)

    criterion = nn.BCELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

    # Guard: BatchNorm requires >1 sample; if a lot is tiny, skip BN by dropping to training mode only
    model.train()
    for _ in range(30):
        optimizer.zero_grad()
        loss = criterion(model(Xt), yt)
        loss.backward()
        optimizer.step()

    model = model.cpu()

    # Test metrics if a held-out split is available for this lot, else fall back to train fit
    metrics = {}
    model.eval()
    with torch.no_grad():
        if df_test is not None and len(df_test) > 0:
            X_test = scaler.transform(df_test[feature_cols].fillna(0).values.astype('float32'))
            y_test = df_test['YIELD_GOOD'].values.astype('float32')
            probs  = model(torch.tensor(X_test)).squeeze().numpy()
            preds  = (probs >= 0.5).astype(int)
            metrics = {
                'test_accuracy':  float(accuracy_score(y_test, preds)),
                'test_precision': float(precision_score(y_test, preds, zero_division=0)),
                'test_recall':    float(recall_score(y_test, preds, zero_division=0)),
                'test_f1':        float(f1_score(y_test, preds, zero_division=0)),
                'test_roc_auc':   float(roc_auc_score(y_test, probs)) if len(set(y_test.tolist())) > 1 else float('nan'),
                'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
            }
        else:
            probs = model(torch.tensor(X_train)).squeeze().numpy()
            preds = (probs >= 0.5).astype(int)
            metrics = {
                'train_accuracy': float(accuracy_score(y_train, preds)),
                'n_train': int(len(X_train)), 'n_test': 0,
            }

    return scaler, model, metrics

print("Shared recipe defined: train_lot_dnn_recipe(df_train, df_test=None) -> (scaler, model, metrics)")


## 📘 Section 4 — Profile One Lot First

Sanity check before launching thousands of jobs. Uses the same recipe MMT will use — so the timing here projects the full-run wall-clock directly.


In [ ]:
# Pick the biggest lot
biggest_lot = session.sql(f"""
    SELECT {PARTITION_COL}
    FROM {TRAINING_STAGING_TABLE}
    GROUP BY {PARTITION_COL}
    ORDER BY COUNT(*) DESC
    LIMIT 1
""").collect()[0][0]

one_lot_sdf = session.table(TRAINING_STAGING_TABLE).filter(col(PARTITION_COL) == biggest_lot)
one_lot_pd  = DataConnector.from_dataframe(one_lot_sdf).to_pandas()
print(f"Profiling lot: {biggest_lot}  |  {len(one_lot_pd):,} rows")

# In-lot 80/20 split
import numpy as np
rng = np.random.default_rng(42)
mask = rng.random(len(one_lot_pd)) < 0.8
df_tr, df_te = one_lot_pd[mask].reset_index(drop=True), one_lot_pd[~mask].reset_index(drop=True)

t0 = time.time()
scaler_profile, model_profile, metrics_profile = train_lot_dnn_recipe(df_tr, df_te)
fit_sec = time.time() - t0

print(f"Fit time: {fit_sec:.2f} sec")
print(f"Profile metrics: {metrics_profile}")
print(f"\nProjected wall-clock for all {total_lots:,} lots:")
for n_parallel in [1, 2, 4, 8]:
    projected_min = (total_lots * fit_sec / n_parallel) / 60
    print(f"   {n_parallel} parallel workers -> {projected_min:6.1f} min")


## 📘 Section 5 — Scale the Container Runtime Cluster

Request all nodes in `WAFER_TRAINING_POOL` (MAX_NODES=2, one A10G GPU each). Start once at least 1 is ready.


In [ ]:
TOTAL_NODES = 2  # WAFER_TRAINING_POOL has MAX_NODES=2

scale_cluster(
    expected_cluster_size=TOTAL_NODES,
    notebook_name="02_b_ManyModels_ByLotID",
    options={"block_until_min_cluster_size": 1},
)

nodes = get_nodes()
total_cpus = sum(n['cpus'] for n in nodes)
print(f"Cluster: {len(nodes)} nodes, {total_cpus} CPUs total")
for i, n in enumerate(nodes):
    print(f"   Node {i}: {n['name']}  cpus={n['cpus']}")


## 📘 Section 6 — Export Partitioned Parquet to Stage

The performance optimization. Instead of having every worker read through a warehouse (bottlenecked on warehouse concurrency), we:

1. Export the staging table to parquet **once** — `COPY INTO @stage PARTITION BY LOT_ID` writes one file per lot
2. Point MMT at the stage — workers read their parquet file directly, in parallel

Result: data-load time per worker drops from ~20s (warehouse `SELECT`) to sub-second (direct parquet read).


In [ ]:
# Ensure stages exist
session.sql(f"CREATE STAGE IF NOT EXISTS {MODEL_STAGE}").collect()
session.sql(f"CREATE STAGE IF NOT EXISTS {DATA_STAGE}").collect()

# Clear any previous export
try:
    session.sql(f"REMOVE @{DATA_STAGE}/lot_data/").collect()
except Exception:
    pass

# Export partitioned parquet — one file per LOT_ID
session.sql(f'''
    COPY INTO @{DATA_STAGE}/lot_data/
    FROM (SELECT * FROM {TRAINING_STAGING_TABLE})
    PARTITION BY {PARTITION_COL}
    FILE_FORMAT = (TYPE = 'PARQUET')
    HEADER = TRUE
    MAX_FILE_SIZE = 268435456
''').collect()

# Verify
rows = session.sql(f"LIST @{DATA_STAGE}/lot_data/").collect()
print(f"Exported {len(rows)} parquet file(s) to @{DATA_STAGE}/lot_data/")
for r in rows[:5]:
    print(f"   {r['name']}  size={r['size']:,} bytes")
if len(rows) > 5:
    print(f"   ... and {len(rows)-5} more")


## 📘 Section 7 — Run MMT via `run_from_stage`

The training function is a thin wrapper around the shared recipe.

`ExecutionOptions`:
- `num_cpus_per_worker=2` — 2 of the 8 vCPUs per worker
- `num_gpus_per_worker=1` — one A10G per worker
- `use_head_node=True` — include the notebook's node

The polling loop below replaces `training_run.wait()` with real-time progress (throughput, ETA, stall detection).


In [ ]:
def train_fn_mmt(data_connector, context):
    """MMT wrapper: in-partition 80/20 split so each lot's accuracy is held-out."""
    import time
    import numpy as np
    t0 = time.time()
    df = data_connector.to_pandas()
    lot_id = context.partition_id

    # Deterministic 80/20 split per-lot
    rng = np.random.default_rng(42)
    mask = rng.random(len(df)) < 0.8
    df_train = df[mask].reset_index(drop=True)
    df_test  = df[~mask].reset_index(drop=True)
    print(f"[MMT] Lot {lot_id}: loaded {len(df):,} rows in {time.time()-t0:.1f}s  "
          f"(train={len(df_train):,}, test={len(df_test):,})")

    scaler, model, metrics = train_lot_dnn_recipe(df_train, df_test if len(df_test) else None)
    print(f"[MMT] Lot {lot_id}: metrics={metrics}")
    return (scaler, model, metrics)   # pickle the tuple with metrics for later inspection


## 📘 Section 8 — Inspect Per-Lot Results

Spot-check one model. Lots that failed or trained with low accuracy are candidates to investigate.


In [ ]:
# Pull a few lots' models and check their training + test metrics
lot_ids = [k for k, v in training_run.partition_details.items() if "DONE" in str(v.status)]
print(f"Successful lots: {len(lot_ids):,}")
print(f"Failed lots:     {sum(1 for v in training_run.partition_details.values() if 'FAIL' in str(v.status))}")
print(f"\nSample lot IDs trained: {lot_ids[:5]}")

# Load one to verify
sample_lot = lot_ids[0]
bundle = training_run.get_model(sample_lot)
if len(bundle) == 3:
    scaler, model, metrics = bundle
else:
    scaler, model = bundle; metrics = {}

print(f"\nLoaded model for {sample_lot}:")
print(f"   Scaler : {type(scaler).__name__}")
print(f"   Model  : {type(model).__name__}")
print(f"   Metrics: {metrics}")
print(model)


## 📘 Section 9 — Run Distributed Inference (MMI)

Load each trained model and predict on the full dataset in parallel. Results write to `WAFER_YIELD_DEMO.INFERENCE.LOT_PREDICTIONS`.


In [ ]:
def predict_fn_mmt(data_connector, model_tuple, context):
    """MMI inference. Unpacks the (scaler, model, metrics) tuple saved during training."""
    import torch
    import pandas as pd
    from snowflake.snowpark.context import get_active_session

    # model_tuple is now (scaler, model, metrics) — metrics are ignored here
    if len(model_tuple) == 3:
        scaler, model, _ = model_tuple
    else:
        scaler, model = model_tuple  # backwards compat

    df = data_connector.to_pandas()
    lot_id = context.partition_id

    exclude_cols = ['WAFER_ID', 'LOT_ID', 'FIRST_TIMESTAMP', 'YIELD_GOOD',
                    'YIELD_SCORE', 'DOMINANT_DEFECT_TYPE']
    feature_cols = [c for c in df.columns if c.upper() not in [x.upper() for x in exclude_cols]]

    X = df[feature_cols].fillna(0).values.astype('float32')
    X_scaled = scaler.transform(X)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        probs  = model(X_tensor).squeeze().numpy()
        labels = (probs > 0.5).astype(int)

    results = pd.DataFrame({
        'WAFER_ID': df['WAFER_ID'].values,
        'LOT_ID':   df['LOT_ID'].values,
        'PREDICTED_YIELD': labels,
        'PREDICTION_PROB': probs,
    })
    sess = get_active_session()
    sess.create_dataframe(results).write.mode("append").save_as_table(
        "WAFER_YIELD_DEMO.INFERENCE.LOT_PREDICTIONS"
    )
    return len(results)


## 📘 Summary

### What We Covered

| Topic | Snowflake API |
|---|---|
| **Partitioned data export** | `COPY INTO @stage PARTITION BY LOT_ID` |
| **Cluster scaling** | `scale_cluster()` |
| **Parallel training** | `ManyModelTraining` + `run_from_stage()` |
| **Resource config** | `ExecutionOptions(num_cpus/gpus_per_worker)` |
| **Model persistence** | `TorchSerde` → MODEL_STAGE |
| **Parallel inference** | `ManyModelInference` over all partitions |

### Key Takeaway

Training one DNN per `LOT_ID` is fully parallelizable — MMT handles the orchestration, and `run_from_stage` removes the warehouse bottleneck so wall-clock is limited only by your GPU count.
